In [ ]:
# in order for gui to work all cells must be initially ran from top to bottom, 
# after that it is enough to keep rerunning the last cell that is generating the button
# the first run of this cell might take around 3 minutes (it goes through all the data)

import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display
import sounddevice as sd
import librosa

from src.resnet_model import ResNet18
from preprocessing_pipeline.preprocessing import preprocess_data
from preprocessing_pipeline.util import fix_len, MEAN, STD  # uses dataset stats :contentReference[oaicite:6]{index=6}


SR = 22050
CLIP_LEN = 3
MODEL_PATH = "./models/id_453_resnet_trial_221_32_1_dropout_0.5_adamw_lr_0.0005_wd_0.01_clip_length_3_increased_aug_append.pth"

THRESHOLD = 0.50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


net = ResNet18().to(DEVICE)
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
net.load_state_dict(ckpt["net_state_dict"])
net.eval()

Directory ./data already exists. Skipping download.


ResNet(
  (conv1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_runn

In [2]:
def clip_to_spectrogram(clip: np.ndarray, sr: int, clip_len: int) -> np.ndarray:
    clip = fix_len(clip, sr * clip_len)

    S = librosa.feature.melspectrogram(y=clip, sr=sr, n_mels=128)

    S_log = librosa.power_to_db(S, ref=np.max)

    S_norm = (S_log - MEAN) / STD
    return S_norm.astype(np.float32)

In [3]:
def record_audio(seconds=10, sr=SR):
    audio = sd.rec(int(seconds * sr), samplerate=sr, channels=1, dtype="float32")
    sd.wait()
    return audio.squeeze()

@torch.no_grad()
def predict_10s(wave: np.ndarray, sr=SR, clip_len=CLIP_LEN, threshold=THRESHOLD):
    clips = preprocess_data(wave, sr, clip_length=clip_len, rms=True, augment=False, label=0)

    if len(clips) == 0:
        return {"error": "No voiced clips after trimming/silence removal."}

    specs = [clip_to_spectrogram(c, sr, clip_len) for c in clips]

    x = torch.tensor(np.stack(specs), dtype=torch.float32)
    x = x.unsqueeze(1).to(DEVICE)

    logits = net(x)
    probs = torch.softmax(logits, dim=1)[:, 1]

    p_final = probs.mean().item()
    decision = 1 if p_final > threshold else 0

    return {
        "n_clips": len(clips),
        "p_segments": probs.cpu().numpy(),
        "p_final": p_final,
        "decision": decision,
        "threshold": threshold
    }

In [ ]:
btn = widgets.Button(description="Record")
out = widgets.Output()

def on_click(_):
    with out:
        out.clear_output()
        print("Recording 10 seconds... speak now.")
        wave = record_audio(10, SR)

        print("Running model...")
        res = predict_10s(wave, SR, CLIP_LEN, THRESHOLD)

        if "error" in res:
            print("ERROR:", res["error"])
            return

        print(f"Clips used: {res['n_clips']}")
        print(f"Mean prob(class1='allowed'): {res['p_final']:.4f}")
        print(f"Threshold: {res['threshold']}")
        print("Decision:", "ALLOWED (1)" if res["decision"] == 1 else "REJECTED (0)")

btn.on_click(on_click, remove=True)
btn.on_click(on_click)
display(btn, out)


# Mean_prob is the confidence of the machine that it should mark the speaker as "allowed"


Button(description='Record', style=ButtonStyle())

Output()